# Notebook 17 — Modello base vs modello fine-tuned

**Modulo:** 17 — Preaddestramento e introduzione al fine-tuning
**Tempo stimato:** 1-1,5 ore (esecuzione su CPU, non serve GPU)

## Cosa costruirai

Al termine di questo notebook avrai:
- Un modello base preaddestrato (T5-small) caricato ed eseguito localmente
- Osservato "dal vivo" perché un modello base non segue istruzioni
- La sua controparte fine-tuned per seguire istruzioni (Flan-T5-small),
  caricata ed eseguita sugli stessi prompt
- Una funzione di confronto riutilizzabile tra modello base e modello
  specializzato
- Le tue osservazioni scritte sulle differenze osservate

## Prerequisiti

```bash
pip install transformers sentencepiece accelerate python-dotenv
```

Non è richiesta nessuna GPU: T5-small (~60M parametri) e Flan-T5-small
(~80M parametri) girano tranquillamente su CPU in pochi secondi per
generazione. Non serve nemmeno una chiave API: scarichiamo modelli
pubblici da Hugging Face, non chiamiamo nessuna API a pagamento.

Se in futuro userai modelli "gated" (che richiedono approvazione) su
Hugging Face, ti servirà un account gratuito e un access token — vedi
Modulo 17, § 7 della dispensa. Per i modelli di questo notebook non serve.

---
## Cella 0 — Setup

Verifichiamo che le librerie siano installate e carichiamo un eventuale
token Hugging Face da `.env` (opzionale per questo notebook: i modelli che
useremo sono pubblici).

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN")  # opzionale per questo notebook

def verifica_chiave(nome: str, valore: str | None) -> None:
    if valore:
        print(f"OK {nome}: {valore[:8]}...")
    else:
        print(f"-- {nome}: non impostata (va bene, non serve per i modelli pubblici di questo notebook)")

verifica_chiave("HUGGINGFACE_TOKEN", HF_TOKEN)

import transformers
print(f"\ntransformers versione: {transformers.__version__}")

---
## Cella 1 — Carichiamo il modello base: T5-small

T5 (*Text-to-Text Transfer Transformer*) è un modello encoder-decoder
preaddestrato su un volume enorme di testo generico, senza specializzazione
su nessun compito in particolare (Modulo 17, § 2 e § 5). Usiamo la versione
`small` (~60M parametri) per restare leggeri su CPU.

Il flusso per generare testo con un modello Hugging Face è sempre lo
stesso: tokenizzare il prompt, generare token con il modello, decodificare
il risultato in testo.

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer_t5 = T5Tokenizer.from_pretrained("t5-small")
model_t5 = T5ForConditionalGeneration.from_pretrained("t5-small")

def genera(tokenizer, model, prompt: str, max_new_tokens: int = 50) -> str:
    """Tokenizza il prompt, genera testo, decodifica il risultato."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids
    output_ids = model.generate(input_ids, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("Modello base T5-small caricato.")
print(f"Numero di parametri: {model_t5.num_parameters():,}")

---
## Cella 2 — Il modello base con prompt semplici

**Prima di eseguire:** T5-small non è stato specializzato a seguire
istruzioni. Con un prompt che è solo l'inizio di una frase, cosa ti aspetti
che generi? Una continuazione naturale, o qualcosa di più strano?

In [ ]:
prompt_semplici = [
    "My name",
    "Today is",
    "Mi chiamo",
]

for p in prompt_semplici:
    risultato = genera(tokenizer_t5, model_t5, p)
    print(f"Prompt: {p!r}")
    print(f"  -> {risultato!r}\n")

**Cosa osservare:** con "My name" e "Today is" il modello genera una
continuazione plausibile ma spesso ripetitiva, senza un punto di arresto
"naturale" da un punto di vista umano. Con "Mi chiamo" (italiano) il
comportamento tende a peggiorare: T5 è preaddestrato prevalentemente in
inglese.

---
## Cella 3 — Il modello base di fronte a istruzioni esplicite

Ora proviamo a **dare un'istruzione** al modello, invece di lasciargli solo
continuare una frase: chiedergli di riassumere un testo o tradurre una
frase. Un modello che sa seguire istruzioni dovrebbe eseguire il compito
richiesto, non limitarsi a continuare il prompt.

**Prima di eseguire:** secondo te, T5-small (non specializzato) eseguirà
l'istruzione o continuerà a comportarsi come nella Cella 2?

In [ ]:
testo_hubble = (
    "Il telescopio Hubble, lanciato nello spazio nel 1990, ha fornito "
    "immagini straordinarie dell'universo e ha aiutato gli scienziati "
    "a comprendere meglio la cosmologia."
)

prompt_istruzioni = [
    f"Riassumi il seguente testo:\n\n{testo_hubble}",
    "Translate to Italian: 'How are you?'",
]

for p in prompt_istruzioni:
    risultato = genera(tokenizer_t5, model_t5, p, max_new_tokens=60)
    print(f"Prompt: {p!r}")
    print(f"  -> {risultato!r}\n")

**Cosa osservare:** con tutta probabilità il modello non riassume né
traduce correttamente. Continua a generare testo come se l'istruzione
fosse solo un prompt da estendere, non un comando da eseguire. Questo è
esattamente il comportamento descritto nel Modulo 17, § 2: un modello
preaddestrato genera testo coerente, ma non è stato specializzato a capire
e seguire istruzioni.

---
## Cella 4 — Carichiamo il modello fine-tuned: Flan-T5-small

**FLAN-T5** (*Fine-tuned LAnguage Net*) è una versione di T5 a cui è stato
applicato instruction fine-tuning su oltre mille compiti diversi (Modulo
17, § 2 caso pratico). Stessa architettura di base, stesso flusso di
caricamento — cambia solo il checkpoint.

In [ ]:
tokenizer_flan = T5Tokenizer.from_pretrained("google/flan-t5-small")
model_flan = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")

print("Modello fine-tuned Flan-T5-small caricato.")
print(f"Numero di parametri: {model_flan.num_parameters():,}")

---
## Cella 5 — Esperimento guidato: confronto diretto, stesso prompt, due modelli

Lanciamo lo **stesso set di prompt** su entrambi i modelli e mettiamo le
risposte una accanto all'altra. È il confronto che conta davvero: stessa
architettura di base, stessa dimensione, l'unica differenza è il
fine-tuning.

In [ ]:
PROMPT_TEST = [
    ("Continuazione (EN)", "My name"),
    ("Continuazione (IT)", "Mi chiamo"),
    ("Istruzione: riassumi (IT)", f"Riassumi il seguente testo:\n\n{testo_hubble}"),
    ("Istruzione: traduci (EN->IT)", "Translate to Italian: 'How are you?'"),
]

print(f"{'CASO':32} | {'T5-small (base)':45} | {'Flan-T5-small (fine-tuned)'}")
print("-" * 130)

for etichetta, prompt in PROMPT_TEST:
    out_base = genera(tokenizer_t5, model_t5, prompt, max_new_tokens=60)
    out_flan = genera(tokenizer_flan, model_flan, prompt, max_new_tokens=60)
    print(f"{etichetta:32} | {out_base[:43]:45} | {out_flan[:60]}")

### Esperimento aggiuntivo: classificazione few-shot vs zero-shot

Il Modulo 17 (§ 4) mostra che il prompt engineering few-shot può orientare
parzialmente un modello base, ma resta meno affidabile del fine-tuning.
Proviamo una classificazione di sentiment: prima chiediamo direttamente al
modello base (zero-shot), poi gli forniamo due esempi nel prompt
(few-shot), infine chiediamo la stessa cosa al modello fine-tuned
(zero-shot, senza esempi).

In [ ]:
recensione = "Il prodotto è arrivato rotto e il servizio clienti non ha mai risposto."

# 1. Modello base, zero-shot
prompt_zero_shot = f"Sentimento? Recensione: {recensione}"
print("T5-small, zero-shot:")
print(" ", genera(tokenizer_t5, model_t5, prompt_zero_shot, max_new_tokens=10))

# 2. Modello base, few-shot (due esempi nel prompt)
prompt_few_shot = f"""Recensione: Bellissimo prodotto, arrivato in perfette condizioni.
Sentimento: positivo

Recensione: Non funziona, è già rotto dopo un giorno.
Sentimento: negativo

Recensione: {recensione}
Sentimento:"""
print("\nT5-small, few-shot (2 esempi):")
print(" ", genera(tokenizer_t5, model_t5, prompt_few_shot, max_new_tokens=10))

# 3. Modello fine-tuned, zero-shot
print("\nFlan-T5-small, zero-shot:")
print(" ", genera(tokenizer_flan, model_flan, prompt_zero_shot, max_new_tokens=10))

**Le tue osservazioni** (modifica questa cella):

- Con il modello base e zero-shot ho ottenuto: ...
- Con il modello base e few-shot (2 esempi) ho ottenuto: ...
- Con il modello fine-tuned e zero-shot ho ottenuto: ...
- Il few-shot ha aiutato il modello base? Quanto, rispetto al fine-tuning? ...
- In quale scenario useresti il few-shot invece del fine-tuning, e perché
  (vedi Modulo 17, § 6)? ...

---
## Esercizio — completa il codice

**TODO:** scrivi due prompt tuoi (uno di semplice continuazione, uno con
un'istruzione esplicita, per esempio "Elenca tre pianeti del sistema
solare" o "Traduci in inglese: ...") e confrontali sui due modelli usando
la funzione `genera` già definita.

In [ ]:
# TODO: scrivi qui i tuoi due prompt
mio_prompt_continuazione = ""  # es. "Il tempo oggi è"
mio_prompt_istruzione = ""     # es. "Elenca tre pianeti del sistema solare"

# TODO: chiama genera() con entrambi i modelli su entrambi i prompt e stampa i risultati


---
## Sfida finale — funzione di confronto riutilizzabile

Costruisci una funzione `confronta_modelli` che:

- accetta una lista di prompt e i due modelli/tokenizer (base e fine-tuned)
- genera l'output di entrambi per ogni prompt
- restituisce i risultati in una struttura dati (lista di dizionari) invece
  di limitarsi a stamparli, così puoi riusarla anche fuori da questo
  notebook

Completa i punti marcati con `# TODO`.

In [ ]:
def confronta_modelli(
    prompts: list[str],
    tok_base, mod_base,
    tok_ft, mod_ft,
    max_new_tokens: int = 50
) -> list[dict]:
    """Confronta modello base e modello fine-tuned sugli stessi prompt.

    Restituisce una lista di dizionari con chiavi:
      prompt, output_base, output_fine_tuned
    """
    risultati = []
    for p in prompts:
        # TODO: genera l'output del modello base con tok_base/mod_base
        out_base = None

        # TODO: genera l'output del modello fine-tuned con tok_ft/mod_ft
        out_ft = None

        risultati.append({
            "prompt": p,
            "output_base": out_base,
            "output_fine_tuned": out_ft,
        })
    return risultati


# Prova la tua funzione con un piccolo set di prompt
prompt_sfida = [
    "Traduci in italiano: 'The weather is nice today.'",
    "Elenca tre pianeti del sistema solare.",
]

risultati_sfida = confronta_modelli(
    prompt_sfida, tokenizer_t5, model_t5, tokenizer_flan, model_flan
)

for r in risultati_sfida:
    print(f"Prompt: {r['prompt']}")
    print(f"  Base:        {r['output_base']}")
    print(f"  Fine-tuned:  {r['output_fine_tuned']}\n")

### Estensione opzionale (se hai tempo ed è disponibile una GPU/Colab)

Il Modulo 17 (§ caso pratico) mostra che aumentare i parametri del modello
fine-tuned (Flan-T5-**large**, ~780M parametri, o versioni ancora più
grandi) migliora ulteriormente le risposte, ma richiede più RAM e più
tempo di generazione. Su CPU locale può essere lento; su Colab gratuito con
runtime GPU (T4) è gestibile.

Imposta `ESEGUI_MODELLO_GRANDE = True` solo se hai una GPU disponibile o
sei disposto ad aspettare qualche minuto su CPU.

In [ ]:
ESEGUI_MODELLO_GRANDE = False  # richiede più RAM/tempo — consigliato Colab con GPU

if ESEGUI_MODELLO_GRANDE:
    tokenizer_flan_large = T5Tokenizer.from_pretrained("google/flan-t5-large")
    model_flan_large = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large")

    for etichetta, prompt in PROMPT_TEST:
        out = genera(tokenizer_flan_large, model_flan_large, prompt, max_new_tokens=60)
        print(f"{etichetta}: {out}\n")
else:
    print("Estensione disattivata. Imposta ESEGUI_MODELLO_GRANDE = True per eseguirla.")

---
## Autovalutazione della settimana

Rispondi in questa cella prima di chiudere il notebook:

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 17 — Modulo 17 del Corso Completo di IA Generativa*